# 04 -- Session and news analysis

Paired script: `analysis/join_news_events.py` for the NEWS half. Independently recomputes
`NEWS_BLACKOUT` status (per `NewsManager.mqh`/section 10) for every journal decision,
directly useful given every real journal record's `news_state` is currently always empty
(the live EA never sets it -- see `analysis/schema.py`'s docstring).

**Fixed, 2026-07-21 Codex review finding:** this notebook previously performed only the news
join and no SESSION analysis at all, despite the name. It now also groups synthetic decisions
by trading session (derived from the decision's own UTC hour, matching a common London/
New York/Asia session convention) and reports win rate by session -- the second half of what
this notebook's name promises.

**Uses clearly-labelled SYNTHETIC journal/news/trade fixtures.** Real-data run: PENDING.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.join_news_events import run as run_news_join

## News-blackout join

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_news_demo_"))

decision = {
    "signal_id": "sig-1", "timestamp_utc": "2026-07-21T14:05:30Z", "symbol": "XAUUSD",
    "market_family": "METAL", "intraday_mode": "SCALP", "regime": "REGIME_TRENDING_UP",
    "regime_confidence": 72.5, "direction": "BUY", "strategy": "TrendFollowingStrategy",
    "setup": "TrendlinePullback", "candlestick_pattern": None, "chart_pattern": None,
    "score": 68.0, "score_breakdown": {}, "entry": 2350.55, "stop": 2345.10,
    "targets": [2361.45], "risk_percent": 0.3, "news_state": "", "session_state": "",
    "reasons_passed": [], "reasons_rejected": [], "ea_version": "1.01", "git_commit": "abc",
}
(tmp_dir / "decisions_20260721.jsonl").write_text(json.dumps(decision) + "\n", encoding="utf-8")

pd.DataFrame([{
    "event_id": "e-nfp", "event_name": "NFP", "currency": "USD", "importance": 2,
    "scheduled_utc": "2026-07-21T14:10:00Z",  # 4m30s after the decision -- inside the window
}]).to_csv(tmp_dir / "news.csv", index=False)

news_result = run_news_join(tmp_dir, tmp_dir / "news.csv", currency="USD", before_minutes=15,
                             after_minutes=15, min_importance=2, repo_path=PROJECT_ROOT.parents[1])

print(f"n_decisions      = {news_result.n_decisions}")
print(f"n_in_blackout    = {news_result.n_in_blackout}")

assert news_result.n_in_blackout == 1
assert news_result.joined.iloc[0]["triggering_event_id"] == "e-nfp"

## Time-of-day breakdown (real, derived dimensions -- not an invented session bucket)

**Fixed, 2026-07-22 Codex review finding:** this cell previously defined a made-up
fixed-UTC-hour `session_for_hour` Asia/London/New-York bucketing function, computed inline
in the notebook rather than in a paired script -- and that bucketing was NOT sourced from
this project's actual session logic at all. `SessionManager.mqh` has **no fixed Asia/
London/New-York UTC-hour concept anywhere** -- it only ever computes session-time-remaining
from the BROKER'S OWN per-symbol session calendar (`SymbolInfoSessionTrade`, a live-MT5 API
this Python layer cannot call offline). Inventing a fixed-UTC-hour substitute was not
"porting the broker-session logic" as the master prompt requires; it was fabricating a
different, un-validated one.

This cell instead uses the real, paired `analysis/performance_breakdown.py` pipeline's
`hour_of_day`/`day_of_week` dimensions -- genuinely DERIVED from each decision's own
`entry_time` (no invented categorization). A true SESSION breakdown (Asia/London/NY-style)
would require either porting `SessionManager.mqh`'s real broker-session-table logic (needs a
real exported session table -- see `TASK-037_MT5_EXPORT_BRIDGE.md`) or consuming the
journal schema's own `session_state` field once the live EA populates it (see
`TASK-036_JOURNAL_PRODUCER_COMPLETION.md`) -- neither of which this notebook fabricates a
substitute for.

In [ ]:
import tempfile

from analysis.performance_breakdown import run as run_breakdown

# Trades entering at the same real UTC hour, three per hour, so the
# hour_of_day grouping below has genuine multi-trade statistics --
# hour_of_day is DERIVED from entry_time, not an invented session label.
synthetic_trades = pd.DataFrame({
    "trade_id": [f"s{i}" for i in range(9)],
    "entry_time": (
        ["2026-07-21T02:00:00Z"] * 3 + ["2026-07-21T09:00:00Z"] * 3 + ["2026-07-22T15:00:00Z"] * 3
    ),
    "profit": [10.0, -5.0, 10.0, 10.0, 10.0, -5.0, -5.0, -5.0, 10.0],
})
breakdown_dir = Path(tempfile.mkdtemp(prefix="themba_hourofday_demo_"))
trades_csv = breakdown_dir / "trades.csv"
synthetic_trades.to_csv(trades_csv, index=False)

performance_by_hour = run_breakdown(trades_csv, ["hour_of_day"])
print(performance_by_hour[["hour_of_day", "n_trades", "win_rate", "expectancy_dollars"]])

assert len(performance_by_hour) == 3
hour2_row = performance_by_hour[performance_by_hour["hour_of_day"] == 2].iloc[0]
assert abs(hour2_row["win_rate"] - (2.0 / 3.0)) < 1e-9  # 2 wins of 3
hour15_row = performance_by_hour[performance_by_hour["hour_of_day"] == 15].iloc[0]
assert abs(hour15_row["win_rate"] - (1.0 / 3.0)) < 1e-9  # 1 win of 3

## Real-data run: PENDING

Requires a real journal (batched runtime verification, TASK-025+), a real news-event export,
and enough real decisions spanning multiple sessions -- none exist yet. A real session
breakdown should also use `SessionManager.mqh`'s own broker-session-time logic (ported to
Python, not yet done) rather than this notebook's simplified fixed UTC-hour buckets.